# IMPORTS

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
print(torch.__version__)
print(f"GPU disponible: {torch.cuda.is_available()}")
import requests
import mlflow
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import datetime

2.11.0+cpu
GPU disponible: False


# Data load

In [2]:
prices  = pd.read_parquet("../data/features/", engine="pyarrow")

# Division

In [3]:

fs_pdf = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
fs_pdf.info()
fs_pdf['date'] = pd.to_datetime(fs_pdf['date'])
FEATURE_COLS = ["close", "volume", "log_return", "rsi", "macd", 
                "bb_position", "volatility", "vol_ratio"]
TARGET_COL = "target"
SEQUENCE_LENGTH = 20

# Split temporal
train = fs_pdf[fs_pdf["date"] < "2023-01-01"]
val   = fs_pdf[(fs_pdf["date"] >= "2023-01-01") & (fs_pdf["date"] < "2024-01-01")]
test  = fs_pdf[fs_pdf["date"] >= "2024-01-01"]

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40220 entries, 0 to 40219
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         40220 non-null  datetime64[ms]
 1   week         40220 non-null  UInt32        
 2   close        40220 non-null  float64       
 3   volume       40220 non-null  int64         
 4   log_return   40200 non-null  float64       
 5   rsi          39960 non-null  float64       
 6   macd         39720 non-null  float64       
 7   bb_position  39840 non-null  float64       
 8   volatility   39820 non-null  float64       
 9   vol_ratio    40040 non-null  float64       
 10  target       40220 non-null  int64         
 11  ticker       40220 non-null  category      
 12  year         40220 non-null  category      
dtypes: UInt32(1), category(2), datetime64[ms](1), float64(7), int64(2)
memory usage: 3.3 MB
Train: 30200 filas
Val:   5000 filas
Test:  5020 filas


# Normalization

In [4]:
global_features = [ 'log_return',
 'rsi',
 'macd',
 'bb_position',
 'volatility',
 'vol_ratio']
ticker_features = ['close','volume']

In [6]:
scaler = StandardScaler()
for col in global_features:
    train[col] = scaler.fit_transform(train[[col]])
    test[col] = scaler.fit_transform(test[[col]])
    val[col] = scaler.fit_transform(val[[col]])



C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2177470906.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[col] = scaler.fit_transform(train[[col]])
C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2177470906.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[col] = scaler.fit_transform(test[[col]])
C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2177470906.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_in

In [7]:
tickers = train['ticker'].unique()

In [8]:
scalers_by_ticker = {}
for ticker in tickers:
    scaler = StandardScaler().fit(train[train['ticker'] == ticker][['close','volume']])
    scalers_by_ticker[ticker] = scaler
for ticker in tickers:
    train.loc[train['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(train[train['ticker'] == ticker][ticker_features])

C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2679068276.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.0068353  -0.57890358 -0.49862233 ... -0.56087669 -0.7425396
 -0.71771064]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.loc[train['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(train[train['ticker'] == ticker][ticker_features])


In [9]:
scalers_by_ticker = {}
for ticker in tickers:
    scaler = StandardScaler().fit(test[test['ticker'] == ticker][['close','volume']])
    scalers_by_ticker[ticker] = scaler
for ticker in tickers:
    test.loc[test['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(test[test['ticker'] == ticker][ticker_features])

C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2184533841.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 8.24402880e-01  4.03053159e-02  4.82251393e-01  1.69452023e-01
  6.40814423e-02 -4.66898086e-01 -3.38210616e-01 -2.62143297e-01
 -5.43893651e-01  2.74435021e-01 -3.21127632e-01  6.78394665e-01
  3.81916140e-01  9.63062360e-02 -4.82733637e-01 -1.15483681e-01
 -7.66992819e-02 -4.09828868e-01 -3.26723164e-01 -4.29143834e-02
 -5.56688096e-02  2.51062763e-01  1.47785551e+00  4.06858273e-01
 -4.45760132e-01 -1.21746899e-01 -5.28123240e-01 -3.91550564e-01
 -5.01419067e-01 -2.10892022e-02 -8.29397008e-02  2.68946969e-01
 -2.41816337e-01 -1.14366529e-01 -5.14789067e-01 -1.59098216e-01
 -3.92706800e-01 -5.31204365e-01 -9.30885241e-02 -2.67826768e-01
  2.58950016e+00  5.33695765e-01  7.92529842e-01  1.23620865e+00
  3.71646807e-01  4.75134840e-01  6.21761886e-01  9.64886282e-02
  8.62583799e-02 -1.5269

In [10]:
scalers_by_ticker = {}
for ticker in tickers:
    scaler = StandardScaler().fit(val[val['ticker'] == ticker][['close','volume']])
    scalers_by_ticker[ticker] = scaler
for ticker in tickers:
    val.loc[val['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(val[val['ticker'] == ticker][ticker_features])

C:\Users\migue\AppData\Local\Temp\ipykernel_18596\512928834.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 2.98032476e+00  1.68416349e+00  1.22489871e+00  1.60759588e+00
  6.51760120e-01  2.63281926e-01  5.76713837e-01  6.84936224e-01
 -7.96636444e-02  2.49218143e-01  5.88766093e-01 -5.31419193e-02
  1.18325391e+00  1.26983971e+00  4.06336967e-01  3.70512636e-01
 -2.88400383e-01  6.38518979e-01  2.69992649e-01  3.74749801e-01
  1.03900999e+00  3.33087695e+00  5.36033829e+00  5.99218143e-01
  1.35786795e+00  2.75897635e-01 -1.81231653e-01 -9.98916013e-02
  1.67652710e-01  1.39964638e-01  3.57806774e-01  5.03972075e-01
 -4.47649758e-03 -2.00785067e-02 -4.62721422e-01 -6.10182665e-01
 -2.11517243e-01 -8.01514345e-01 -4.88882538e-01 -2.10987598e-01
 -3.93597028e-01  6.48463921e-01  1.59651276e+00 -1.71376863e-01
 -6.77199746e-01 -3.03698128e-01  5.26763743e-01  1.42179167e+00
  8.15448805e-01  1.01107

## Encoding

In [11]:
ticker_dummies = pd.get_dummies(train["ticker"], prefix="ticker")
train = pd.concat([train, ticker_dummies], axis=1)
ticker_dummies = pd.get_dummies(test["ticker"], prefix="ticker")
test = pd.concat([test, ticker_dummies], axis=1)
ticker_dummies = pd.get_dummies(val["ticker"], prefix="ticker")
val = pd.concat([val, ticker_dummies], axis=1)


In [12]:
val.columns

Index(['date', 'week', 'close', 'volume', 'log_return', 'rsi', 'macd',
       'bb_position', 'volatility', 'vol_ratio', 'target', 'ticker', 'year',
       'ticker_AAPL', 'ticker_ADBE', 'ticker_AMZN', 'ticker_BAC',
       'ticker_BRK-B', 'ticker_DIS', 'ticker_GOOGL', 'ticker_HD', 'ticker_JNJ',
       'ticker_JPM', 'ticker_MA', 'ticker_META', 'ticker_MSFT', 'ticker_NFLX',
       'ticker_NVDA', 'ticker_PG', 'ticker_PYPL', 'ticker_TSLA', 'ticker_UNH',
       'ticker_V'],
      dtype='object')

In [13]:
ticker_cols = []
for col in train.columns:
    if col.startswith('ticker_'): 
        ticker_cols.append(col)


In [14]:
ticker_cols

['ticker_AAPL',
 'ticker_ADBE',
 'ticker_AMZN',
 'ticker_BAC',
 'ticker_BRK-B',
 'ticker_DIS',
 'ticker_GOOGL',
 'ticker_HD',
 'ticker_JNJ',
 'ticker_JPM',
 'ticker_MA',
 'ticker_META',
 'ticker_MSFT',
 'ticker_NFLX',
 'ticker_NVDA',
 'ticker_PG',
 'ticker_PYPL',
 'ticker_TSLA',
 'ticker_UNH',
 'ticker_V']

# Format data for entry

In [15]:
cols = ["close","volume","log_return","rsi","macd","bb_position","volatility","vol_ratio",
        "ticker_AAPL",
        "ticker_ADBE",
        "ticker_AMZN",
        "ticker_BAC",
        "ticker_BRK-B",
        "ticker_DIS",
        "ticker_GOOGL",
        "ticker_HD",
        "ticker_JNJ",
        "ticker_JPM",
        "ticker_MA",
        "ticker_META",
        "ticker_MSFT",
        "ticker_NFLX",
        "ticker_NVDA",
        "ticker_PG",
        "ticker_PYPL",
        "ticker_TSLA",
        "ticker_UNH",
        "ticker_V"]


In [17]:
def collect_window(df, window_size):
    # df debe estar ordenado por date
    context_list = []
    values = df[cols].values.tolist()  # lista de listas
    for i in range(len(df)):
        start = max(0, i - window_size + 1)
        context_list.append(values[start:i+1])
    return pd.Series(context_list, index=df.index)


In [18]:

context_size =20
train["context"] = train.groupby("ticker", group_keys=False).apply(lambda x: collect_window(x, context_size))
test["context"] = test.groupby("ticker", group_keys=False).apply(lambda x: collect_window(x, context_size))
val["context"] = val.groupby("ticker", group_keys=False).apply(lambda x: collect_window(x, context_size))

C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2794467937.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train["context"] = train.groupby("ticker", group_keys=False).apply(lambda x: collect_window(x, context_size))
C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2794467937.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train["context"] = train.groupby("ticker", group_keys=False).apply(lambda x: collect_window(x, context_size))
C:\Users\migue\AppData\Local\Temp\ipykernel_18596\2794467937.py:3: Futur

In [0]:
train.printSchema()

In [19]:

val = val.sort_values(["ticker", "date"]).reset_index(drop=True)
test = test.sort_values(["ticker", "date"]).reset_index(drop=True)
train = train.sort_values(["ticker", "date"]).reset_index(drop=True)

In [0]:
print(f"Shape: {fs_pdf.shape}")
print(f"Tickers: {fs_pdf['ticker'].nunique()}")
print(f"Rango: {fs_pdf['date'].min()} → {fs_pdf['date'].max()}")

In [20]:

train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
val['date'] = pd.to_datetime(val['date'])

In [24]:

train.loc[0]['context']

[[-1.2297463820029273,
  -0.006835301902778168,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  True,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False]]

In [ ]:
def change_format(context_list):
    return np.array([[v for v in d.values()] for d in context_list])

# # Convertir context a arrays (20, 8)
train["context"] = train["context"].map(change_format)
val["context"]   = val["context"].map(change_format)
test["context"]  = test["context"].map(change_format)

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

AttributeError: 'list' object has no attribute 'values'

In [25]:
train = train[train["context"].map(lambda x: x.shape[0]) == 20]
val   = val[val["context"].map(lambda x: x.shape[0]) == 20]
test  = test[test["context"].map(lambda x: x.shape[0]) == 20]

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

AttributeError: 'list' object has no attribute 'shape'

# Training LSTM

## All variables no exogen

In [0]:
#20 del one hot encoding, más 8 de features numéricas

# rnn = torch.nn.LSTM(
#     input_size=28,
#     hidden_size=64,
#     num_layers=2,#más capas, mas overfitting
#     #dropout = 0.2,
#     barch_first = True,# por si le matriz de entrada está girada
# )

class MarketLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,dropout=0.2):
        super(MarketLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.dropout = dropout
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, dropout=dropout,batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(#inicializar los arrays de memoria
                0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(
                0), self.hidden_dim).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # Take last time step
        return out, hn, cn
lstm = MarketLSTM(28,64,2,1,0.3)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm.parameters(),lr=0.001)

In [0]:
print(lstm)

In [0]:


class MarketDataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.tensor(row["context"], dtype=torch.float32)
        y = torch.tensor(row["target"], dtype=torch.float32)
        return x,y


In [0]:
train_dataset = MarketDataset(train)
val_dataset   = MarketDataset(val)
test_dataset  = MarketDataset(test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)#dividir en batches los datos
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [0]:
train["target"].value_counts(normalize=True)

In [0]:
epochs = 100

with mlflow.start_run(run_name="lstm-technical-all-features"):
    mlflow.log_params({
        "input_dim": 28,
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.3,
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": epochs,
        "features": "all"
    })
    for epoch in range(epochs):
        train_loss = 0
        lstm.train()
        for x_batch,y_batch in train_loader:
            optimizer.zero_grad()#reiniciar optimizador para que optimice de acuerdo al batch y epoch actual
            outputs,_,_ = lstm(x_batch)
            outputs = outputs.squeeze(1)#aplastar para que sea un vector y no una matriz de una fila
            loss = criterion(outputs, y_batch)
            train_loss += loss.item()
            loss.backward()#calcular gradientes con backpropagation
            optimizer.step()#actualizar pesos

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item()}')
        lstm.eval()#se modifican los modos del modelo para que no aprenda al insertar datos nuevos
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                outputs,_,_ = lstm(x_batch)
                outputs = outputs.squeeze(1)#aplastar para que sea un vector y no una matriz de una fila
                loss = criterion(outputs, y_batch)
                prob = torch.sigmoid(outputs)
                pred = (prob > 0.5).float()
                correct += (pred == y_batch).sum().item()
                total += y_batch.size(0)
                val_loss += loss.item()

        mlflow.log_metrics({
            "train_loss": train_loss / len(train_loader),
            "val_loss": val_loss / len(val_loader),
            "val_accuracy": correct / total
        }, step=epoch)

        

In [0]:
# Baseline tonto: siempre predice 1
y_true = val["target"].values
baseline_accuracy = y_true.mean()
print(f"Accuracy si siempre predices 1: {baseline_accuracy:.4f}")
print(f"Accuracy si siempre predices 0: {1-baseline_accuracy:.4f}")

# Verificar que el DataLoader mantiene el orden correcto
first_batch_targets = next(iter(val_loader))[1]
print(f"Primeros targets del val_loader: {first_batch_targets[:10]}")
print(f"Primeros targets del DataFrame: {val['target'].values[:10]}")

In [0]:
# Guardar el modelo en MLflow Registry
mlflow.pytorch.log_model(
    lstm,
    artifact_path="market-lstm-technical",
    registered_model_name="market-mood-lstm-technical"
)

In [0]:
epochs = 100
h0, c0 = None, None
for epoch in range(epochs):
    lstm.train()
    optimizer.zero_grad()
    outputs,h0,c0 = lstm(train['context'],h0,c0)
    loss = criterion(outputs, train['target'])
    
    loss.backward()
    optimizer.step()

    h0,c0=h0.detach(),c0.detach()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')